# L8c: The Condition Number of a Matrix
The __condition number__ of a matrix measures how much the solution of a linear problem can change when its data change by a small amount. We used it in the L7c lecture and the L7d lab to compare the normal equations with the singular value decomposition (SVD); here we define it and explain why it matters.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Define the condition number of a matrix:__ The condition number is the ratio of the largest to the smallest singular value of a matrix. We'll interpret it as the ratio of the largest to the smallest amount by which the matrix stretches a vector.
> * __Bound the change in a solution caused by a change in the data:__ For a square linear system, the condition number is the largest factor by which a relative change in the data can grow in the solution. We'll derive this bound, and a similar bound for the least-squares estimate when the model fits the data closely.
> * __Explain why the condition number matters for least squares and ridge regression:__ Forming the normal equations squares the condition number, and adding a ridge penalty to those equations lowers it again. We'll connect the condition number to the number of digits we can trust in a computed estimate and to the scaling of the features.

Throughout, we work with matrices that have at least as many rows as columns. Let's get started!
___

## The condition number
Let $\mathbf{A}\in\mathbb{R}^{n\times{p}}$ with $n\geq{p}$ have the singular value decomposition $\mathbf{A}=\mathbf{U}\;\mathbf{S}\;\mathbf{V}^{\top}$, where $\mathbf{U}$ and $\mathbf{V}$ are orthogonal and the diagonal of $\mathbf{S}$ holds the singular values $\sigma_{1}\geq\sigma_{2}\geq\cdots\geq\sigma_{p}\geq{0}$. When $\sigma_{p}>0$, i.e., when $\mathbf{A}$ has full column rank, the __condition number__ of $\mathbf{A}$ is given by:

$$
\boxed{
\begin{equation*}
\kappa(\mathbf{A}) = \frac{\sigma_{1}}{\sigma_{p}}
\end{equation*}}
$$

If $\mathbf{A}$ is rank deficient, then $\sigma_{p}=0$ and we set $\kappa(\mathbf{A})=\infty$. In Julia, [the `cond(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.cond) computes this ratio; we used it in the L7d lab.

Why is this ratio useful? Any vector can be written as $\mathbf{x}=\mathbf{V}\mathbf{c}$, and then $\lVert\mathbf{x}\rVert_{2}=\lVert\mathbf{c}\rVert_{2}$. Because $\mathbf{U}$ and $\mathbf{V}$ are orthogonal, the length of $\mathbf{A}\mathbf{x}$ is given by:

$$
\begin{align*}
\lVert\mathbf{A}\mathbf{x}\rVert_{2}^{2} &= \lVert\mathbf{S}\mathbf{c}\rVert_{2}^{2} = \sum_{i=1}^{p}\sigma_{i}^{2}c_{i}^{2}\\
\sigma_{p} &\leq \frac{\lVert\mathbf{A}\mathbf{x}\rVert_{2}}{\lVert\mathbf{x}\rVert_{2}} \leq \sigma_{1}\quad\text{(for every $\mathbf{x}\neq\mathbf{0}$)}
\end{align*}
$$

where the second line holds because the sum lies between $\sigma_{p}^{2}\lVert\mathbf{c}\rVert_{2}^{2}$ and $\sigma_{1}^{2}\lVert\mathbf{c}\rVert_{2}^{2}$. The upper bound is reached at $\mathbf{x}=\mathbf{v}_{1}$ and the lower bound at $\mathbf{x}=\mathbf{v}_{p}$, the first and last columns of $\mathbf{V}$. Thus, the condition number is the ratio of the largest to the smallest amount by which $\mathbf{A}$ stretches a vector. The largest stretch is the __matrix 2-norm__, $\lVert\mathbf{A}\rVert_{2}=\sigma_{1}$.

> __Key insights:__
>
> * __Never smaller than one:__ Every matrix has $\kappa\geq{1}$. A matrix with orthonormal columns has every singular value equal to one, so $\kappa=1$: it stretches every direction equally.
> * __Overall scale versus column scale:__ Multiplying the whole matrix by a nonzero number leaves $\kappa$ unchanged, but scaling a single column can change it a lot, as we'll see below.

Next, let's see why this ratio matters when we solve a linear problem.

___

## Sensitivity of a solution to its data
Measured data contain errors, and a computer rounds every number it cannot store exactly. The condition number bounds how much these small changes in the data can change the solution.

### Square linear systems
Consider $\mathbf{A}\mathbf{x}=\mathbf{b}$, where $\mathbf{A}\in\mathbb{R}^{n\times{n}}$ is invertible and $\mathbf{b}\neq\mathbf{0}$. If the right-hand side changes to $\mathbf{b}+\Delta\mathbf{b}$, the solution changes to $\mathbf{x}+\Delta\mathbf{x}$, where $\mathbf{A}\,\Delta\mathbf{x}=\Delta\mathbf{b}$. How large can the relative change in the solution be?

> __Sensitivity bound:__ The inverse $\mathbf{A}^{-1}=\mathbf{V}\mathbf{S}^{-1}\mathbf{U}^{\top}$ has singular values $1/\sigma_{i}$, so its largest stretch is $1/\sigma_{n}$. Using the largest stretch of each matrix:
>
> $$
> \begin{align*}
> \lVert\Delta\mathbf{x}\rVert_{2} &= \lVert\mathbf{A}^{-1}\Delta\mathbf{b}\rVert_{2}\leq\frac{1}{\sigma_{n}}\,\lVert\Delta\mathbf{b}\rVert_{2}\\
> \lVert\mathbf{b}\rVert_{2} &= \lVert\mathbf{A}\mathbf{x}\rVert_{2}\leq\sigma_{1}\,\lVert\mathbf{x}\rVert_{2}
> \end{align*}
> $$
>
> The second line gives $1/\lVert\mathbf{x}\rVert_{2}\leq\sigma_{1}/\lVert\mathbf{b}\rVert_{2}$. Multiplying the two inequalities gives:
>
> $$
> \boxed{
> \begin{equation*}
> \frac{\lVert\Delta\mathbf{x}\rVert_{2}}{\lVert\mathbf{x}\rVert_{2}} \leq \kappa(\mathbf{A})\;\frac{\lVert\Delta\mathbf{b}\rVert_{2}}{\lVert\mathbf{b}\rVert_{2}}
> \end{equation*}}
> $$
>
> Equality holds when $\mathbf{b}$ points along $\mathbf{u}_{1}$ and $\Delta\mathbf{b}$ along $\mathbf{u}_{n}$, the first and last columns of $\mathbf{U}$, so no smaller factor works for every $\mathbf{b}$ and $\Delta\mathbf{b}$.

A relative change in the data can grow by up to a factor of $\kappa(\mathbf{A})$ in the solution. Since $\lVert\mathbf{A}^{-1}\rVert_{2}=1/\sigma_{n}$, many textbooks write $\kappa(\mathbf{A})=\lVert\mathbf{A}\rVert_{2}\lVert\mathbf{A}^{-1}\rVert_{2}$. A similar argument shows that, to first order, a small relative change in the entries of $\mathbf{A}$ grows by the same factor.

#### Example: Two nearly parallel lines
Consider the two-by-two system:

$$
\begin{equation*}
\underbrace{\begin{bmatrix}1 & 1\\ 1 & 1.0001\end{bmatrix}}_{\mathbf{A}}\begin{bmatrix}x_{1}\\ x_{2}\end{bmatrix}=\underbrace{\begin{bmatrix}2\\ 2.0001\end{bmatrix}}_{\mathbf{b}}
\end{equation*}
$$

whose solution is $x_{1}=x_{2}=1$. If we change the second entry of $\mathbf{b}$ by $10^{-4}$, to $2.0002$, the solution becomes $x_{1}=0$ and $x_{2}=2$. The relative changes are:

$$
\begin{equation*}
\frac{\lVert\Delta\mathbf{b}\rVert_{2}}{\lVert\mathbf{b}\rVert_{2}}=\frac{10^{-4}}{\sqrt{2^{2}+2.0001^{2}}}\approx 3.5\times10^{-5}
\qquad
\frac{\lVert\Delta\mathbf{x}\rVert_{2}}{\lVert\mathbf{x}\rVert_{2}}=\frac{\sqrt{(-1)^{2}+1^{2}}}{\sqrt{1^{2}+1^{2}}}=1
\end{equation*}
$$

A relative change of about $3.5\times10^{-5}$ in the data became a relative change of 1 in the solution. The singular values of $\mathbf{A}$ are $\sigma_{1}\approx2.00005$ and $\sigma_{2}\approx5.0\times10^{-5}$, so $\kappa(\mathbf{A})\approx4.0\times10^{4}$, and this change reaches about 70 percent of the bound.

__Why is this system so sensitive?__ Each equation is a line in the $(x_{1},x_{2})$ plane, and these two lines are nearly parallel, so shifting one of them slightly moves their intersection a long way. Equivalently, the two columns of $\mathbf{A}$ point in almost the same direction. In a regression model, the columns are features, and features that are nearly combinations of one another cause the same problem.

### The least-squares estimate
In the L7c lecture, we wrote the least-squares estimate for a data matrix $\hat{\mathbf{X}}\in\mathbb{R}^{n\times{p}}$ with full column rank as $\hat{\mathbf{\theta}}=\sum_{i=1}^{p}\left(\mathbf{u}_{i}^{\top}\mathbf{y}/\sigma_{i}\right)\mathbf{v}_{i}$, where $\mathbf{y}$ holds the observed outputs. The estimate is linear in $\mathbf{y}$, so a change $\Delta\mathbf{y}$ changes the estimate by:

$$
\begin{equation*}
\Delta\hat{\mathbf{\theta}} = \sum_{i=1}^{p}\left(\frac{\mathbf{u}_{i}^{\top}\Delta\mathbf{y}}{\sigma_{i}}\right)\mathbf{v}_{i}
\end{equation*}
$$

The part of $\Delta\mathbf{y}$ along $\mathbf{u}_{p}$ is divided by the smallest singular value. This is the noise amplification we saw in the L7c lecture, and the condition number puts a number on it.

> __Sensitivity bound for least squares:__ Because the $\mathbf{v}_{i}$ are orthonormal, and so are the $\mathbf{u}_{i}$:
>
> $$
> \begin{equation*}
> \lVert\Delta\hat{\mathbf{\theta}}\rVert_{2}^{2}=\sum_{i=1}^{p}\frac{\left(\mathbf{u}_{i}^{\top}\Delta\mathbf{y}\right)^{2}}{\sigma_{i}^{2}}\leq\frac{1}{\sigma_{p}^{2}}\sum_{i=1}^{p}\left(\mathbf{u}_{i}^{\top}\Delta\mathbf{y}\right)^{2}\leq\frac{\lVert\Delta\mathbf{y}\rVert_{2}^{2}}{\sigma_{p}^{2}}
> \end{equation*}
> $$
>
> The fitted values satisfy $\lVert\hat{\mathbf{X}}\hat{\mathbf{\theta}}\rVert_{2}\leq\sigma_{1}\lVert\hat{\mathbf{\theta}}\rVert_{2}$, and for $\hat{\mathbf{\theta}}\neq\mathbf{0}$, combining the two inequalities as before gives:
>
> $$
> \boxed{
> \begin{equation*}
> \frac{\lVert\Delta\hat{\mathbf{\theta}}\rVert_{2}}{\lVert\hat{\mathbf{\theta}}\rVert_{2}} \leq \kappa(\hat{\mathbf{X}})\;\frac{\lVert\Delta\mathbf{y}\rVert_{2}}{\lVert\hat{\mathbf{X}}\hat{\mathbf{\theta}}\rVert_{2}}
> \end{equation*}}
> $$
>
> When the model fits the data well, the fitted values $\hat{\mathbf{X}}\hat{\mathbf{\theta}}$ are close to $\mathbf{y}$, and this is nearly the bound for the square system.

The bound is a worst case. Random noise has only part of its length along $\mathbf{u}_{p}$, so it usually grows by less.

___

## The condition number in least squares and ridge regression
### How many digits can we trust?
A `Float64` number holds about 16 significant decimal digits. The relative gap between 1 and the next larger `Float64` number is the machine epsilon, $\epsilon_{\mathrm{mach}}\approx2.2\times10^{-16}$, returned by [the `eps(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.eps-Tuple{Type{%3C:AbstractFloat}}). Storing the data rounds each entry by a relative amount of at most about this size, and by the bounds above, the solution can change by up to $\kappa$ times as much:

> __Rule of thumb:__ A computed solution can lose about $\log_{10}\kappa$ significant digits, roughly one for each power of ten in the condition number. In `Float64` arithmetic, we can count on only about $16-\log_{10}\kappa$ correct digits.

For the two-by-two example, [the backslash operator](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#Base.:%5C%5C-Tuple{AbstractMatrix,%20AbstractVecOrMat}) returns $x_{1}\approx0.9999999999978$ instead of 1, because $1.0001$ and $2.0001$ cannot be stored exactly in binary. With $\kappa\approx4\times10^{4}$, the rule predicts about 11 correct digits, and the computed $x_{1}$ has between 11 and 12.

Measured data carry far fewer correct digits, and their errors can grow by the same factor. If house prices are known to a relative error of about $10^{-3}$ and $\kappa(\hat{\mathbf{X}})=1000$, the least-squares bound allows an error in the estimate as large as the estimate itself.

So a matrix is __ill-conditioned__ when its condition number times the relative error in the data is too large for the accuracy we need, and __well conditioned__ when that product is small. The [housing example](CHEME-5800-L8c-Example-RLS-SVD-HousingPriceModel-Fall-2026.ipynb) has $\kappa(\hat{\mathbf{X}})\approx42$ for its training split, so it is well conditioned with respect to `Float64` rounding: the rule of thumb predicts fewer than two lost digits.

### The normal equations square the condition number
Substituting the SVD of $\hat{\mathbf{X}}$ into the normal-equations matrix gives:

$$
\begin{equation*}
\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} = \mathbf{V}\mathbf{S}^{\top}\underbrace{\mathbf{U}^{\top}\mathbf{U}}_{=\;\mathbf{I}}\mathbf{S}\mathbf{V}^{\top} = \mathbf{V}\left(\mathbf{S}^{\top}\mathbf{S}\right)\mathbf{V}^{\top}
\quad\Longrightarrow\quad
\boxed{\kappa\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)=\frac{\sigma_{1}^{2}}{\sigma_{p}^{2}}=\kappa(\hat{\mathbf{X}})^{2}}
\end{equation*}
$$

where $\mathbf{S}^{\top}\mathbf{S}$ is the $p\times{p}$ diagonal matrix with entries $\sigma_{i}^{2}$. The right-hand side is an SVD of $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$, so its singular values are the $\sigma_{i}^{2}$.

Solving the normal equations can therefore lose about twice as many digits, $2\log_{10}\kappa(\hat{\mathbf{X}})$. Methods that work with $\hat{\mathbf{X}}$ directly, such as the SVD or [the backslash operator](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#Base.:%5C%5C-Tuple{AbstractMatrix,%20AbstractVecOrMat}), are sensitive to relative changes in $\hat{\mathbf{X}}$ by a factor of at most about $\kappa(\hat{\mathbf{X}})+\kappa(\hat{\mathbf{X}})^{2}\lVert\mathbf{r}\rVert_{2}/\lVert\hat{\mathbf{X}}\hat{\mathbf{\theta}}\rVert_{2}$, where $\mathbf{r}=\mathbf{y}-\hat{\mathbf{X}}\hat{\mathbf{\theta}}$ is the residual vector (Trefethen and Bau, _Numerical Linear Algebra_, Lecture 18).

When $\kappa(\hat{\mathbf{X}})\lVert\mathbf{r}\rVert_{2}\lesssim\lVert\hat{\mathbf{X}}\hat{\mathbf{\theta}}\rVert_{2}$, the first term dominates and these methods lose only about $\log_{10}\kappa(\hat{\mathbf{X}})$ digits, so the SVD helps most when the model fits well.

For the housing example, $\kappa(\hat{\mathbf{X}})\approx42$ and $\kappa(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})\approx1800$, so the rule of thumb predicts more than twelve correct digits for both routes. If $\kappa(\hat{\mathbf{X}})=10^{8}$, the normal-equations matrix has a condition number near $10^{16}$ and no correct digits are guaranteed, while the SVD can keep about eight when the model fits well.

### Where large condition numbers come from
__Nearly dependent features.__ If some combination of the columns nearly cancels, i.e., $\hat{\mathbf{X}}\mathbf{z}\approx\mathbf{0}$ for a unit vector $\mathbf{z}$, then $\sigma_{p}\leq\lVert\hat{\mathbf{X}}\mathbf{z}\rVert_{2}$ is small, because $\sigma_{p}$ is the smallest stretch. Strongly correlated features do this, as do the nearly parallel columns of the two-by-two example. An exact dependence, such as the floor area in square feet and in square meters, gives $\sigma_{p}=0$ and $\kappa=\infty$.

__Features on very different scales.__ Column $j$ of $\hat{\mathbf{X}}$ is $\hat{\mathbf{X}}\mathbf{e}_{j}$, where $\mathbf{e}_{j}$ is the $j$-th unit vector. So $\sigma_{1}$ is at least the length of any column, and $\sigma_{p}$ is at most the length of any column:

$$
\begin{equation*}
\kappa(\hat{\mathbf{X}}) \geq \frac{\lVert\hat{\mathbf{X}}\mathbf{e}_{j}\rVert_{2}}{\lVert\hat{\mathbf{X}}\mathbf{e}_{k}\rVert_{2}}\quad\text{(for any columns $j$ and $k$)}
\end{equation*}
$$

A column with large entries next to the intercept column of ones forces a large condition number. The [housing example](CHEME-5800-L8c-Example-RLS-SVD-HousingPriceModel-Fall-2026.ipynb) divides the area by 1000 for this reason: with the area in square feet, the condition number of the same training data matrix is about 34,000 instead of 42. Putting the features on a common scale, as we'll do in the L8d lab, removes this cause.

### Ridge regression lowers the condition number
The ridge estimate in the [L8c lecture](CHEME-5800-L8c-Lecture-RegularizationAndGeneralization-Fall-2026.ipynb) solves a linear system with the matrix $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\delta\,\mathbf{I}$, where $\delta\geq0$ is the regularization parameter. Writing $\mathbf{I}=\mathbf{V}\mathbf{V}^{\top}$, the same substitution gives $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\delta\,\mathbf{I}=\mathbf{V}\left(\mathbf{S}^{\top}\mathbf{S}+\delta\,\mathbf{I}\right)\mathbf{V}^{\top}$, so the singular values of this matrix are $\sigma_{i}^{2}+\delta$ and its condition number is given by:

$$
\boxed{
\begin{equation*}
\kappa\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\delta\,\mathbf{I}\right)=\frac{\sigma_{1}^{2}+\delta}{\sigma_{p}^{2}+\delta}
\end{equation*}}
$$

For $\delta>0$, this is smaller than $\kappa(\hat{\mathbf{X}})^{2}$ whenever $\sigma_{1}>\sigma_{p}$, decreases toward one as $\delta$ grows, and stays finite even when $\sigma_{p}=0$. The penalty makes the system better conditioned, and in exchange the estimate is biased. In the housing example with $\delta=1$, the condition number falls only from about 1800 to about 1600, because $\delta$ is small compared with $\sigma_{p}^{2}\approx11.5$.

___

## Summary
We defined the condition number of a matrix, derived the bound it places on the change in a solution, and used it to compare the normal equations, the SVD, and ridge regression.

> __Key Takeaways:__
>
> * __The condition number compares stretches:__ We defined the condition number as the ratio of the largest to the smallest singular value, which is the ratio of the largest to the smallest amount by which the matrix stretches a vector. It is never smaller than one, and it is infinite for a rank-deficient matrix.
>
> * __Errors in the data can grow by the condition number:__ We showed that a relative change in the data can grow by up to the condition number in the solution of a square linear system, and by about the same factor in the least-squares estimate when the model fits the data closely. As a rule of thumb, a computed solution can lose about one significant digit for each power of ten in the condition number.
>
> * __Conditioning in least squares and ridge regression:__ We showed that the normal equations square the condition number, which is why methods that work with the data matrix directly are more accurate for ill-conditioned problems that fit the data well. Correlated features and features on very different scales make the condition number large, and the ridge penalty lowers the condition number of the system we solve.

These results give a numerical reason for computing the ridge estimate from the SVD and for putting the features on a common scale before fitting.

___